# DCGAN: Image Generation with Competing Networks

A **GAN** (Generative Adversarial Network) trains two networks against each other:
- **Generator ($G$)**: takes random latent noise $z$ and produces a fake image.
- **Discriminator ($D$)**: takes an image and outputs a score — "how real does this look?"

A **DCGAN** (Deep Convolutional GAN) uses convolutional layers: `ConvTranspose2d` in $G$ to grow
the noise into an image, and `Conv2d` in $D$ to shrink the image into a single real/fake score.

### Key concepts
- **Latent vector $z$**: a list of 100 random numbers (drawn from a bell curve) that acts as a
  "recipe" — same recipe + trained generator = same face; different recipe = different face.
- **Latent space**: the entire cookbook of all possible recipes. Smoothness means nearby recipes
  produce similar-looking images.
- **Batch training**: we process 64 recipes at once so the generator gets smooth, averaged
  feedback instead of noisy feedback from a single example.

This notebook follows the official PyTorch DCGAN tutorial:
https://docs.pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html

In [ ]:
# DCGAN setup: imports, seed, and hyperparameters
# Source: https://docs.pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html

import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.utils.data
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils

# Set random seed for reproducibility
MANUAL_SEED = 999
print("Random Seed:", MANUAL_SEED)
random.seed(MANUAL_SEED)
torch.manual_seed(MANUAL_SEED)
torch.use_deterministic_algorithms(True)

# Number of workers for dataloader
WORKERS = 2

# Batch size during training
BATCH_SIZE = 128

# Spatial size of training images
IMAGE_SIZE = 64

# Number of channels in the training images (3 for RGB)
NUMBER_CHANNELS = 3

# Size of z latent vector (generator input)
NUMBER_Z_DIMENSIONS = 100

# Size of feature maps in generator
NUMBER_GENERATOR_FEATURES = 64

# Size of feature maps in discriminator
NUMBER_DISCRIMINATOR_FEATURES = 64

# Number of training epochs
NUM_EPOCHS = 5

# Learning rate for optimizers
LR = 0.0002

# Beta1 hyperparameter for Adam optimizers
BETA_1 = 0.5

# Number of GPUs available (0 for CPU)
NGPU = 1

In [ ]:
# Resolve project root so paths work regardless of where Jupyter starts
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml")


project_root = find_project_root()
dataroot = str(project_root / "data" / "celeba")
print(f"dataroot: {dataroot}")

In [ ]:
# Dataset and dataloader
# Source: https://docs.pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html

dataset = dset.ImageFolder(
    root=dataroot,
    transform=transforms.Compose([
        transforms.Resize(IMAGE_SIZE),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]),
)

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=WORKERS,
)

# Decide which device to run on
device = torch.device("cuda:0" if (torch.cuda.is_available() and NGPU > 0) else "cpu")

# Plot some training images
real_batch = next(iter(dataloader))
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("Training Images")
plt.imshow(
    np.transpose(
        vutils.make_grid(real_batch[0].to(device)[:64], padding=2, normalize=True).cpu(),
        (1, 2, 0),
    )
)
plt.show()

In [ ]:
# Custom weights initialization
# Source: https://docs.pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html


def weights_init(m: torch.nn.Module) -> None:
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        torch.nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        torch.nn.init.normal_(m.weight.data, 1.0, 0.02)
        torch.nn.init.constant_(m.bias.data, 0)

## Generator layers explained

### ConvTranspose2d — grows the image

```
torch.nn.ConvTranspose2d(
    in_channels=100,    # how many input feature maps
    out_channels=512,   # how many output feature maps
    kernel_size=4,      # 4×4 sliding window
    stride=1,           # stride 1 → output grows slowly (1×1 → 4×4)
    stride=2,           # stride 2 → output doubles (4×4 → 8×8 → ... → 64×64)
    padding=0,          # stride 1 layers use padding=0
    padding=1,          # stride 2 layers use padding=1
    bias=False,         # no extra bias — BatchNorm handles it
)
```

ConvTranspose2d is the generator's engine: it takes a small input and "paints" a larger output
using learned patterns. Each layer doubles the spatial size while halving the channels.

| Layer | Shape out | Channels |
|---|---|---|
| Input z | 1×1 | 100 |
| ConvTranspose2d (stride=1) | 4×4 | 512 |
| ConvTranspose2d (stride=2) | 8×8 | 256 |
| ConvTranspose2d (stride=2) | 16×16 | 128 |
| ConvTranspose2d (stride=2) | 32×32 | 64 |
| ConvTranspose2d (stride=2) | 64×64 | 3 (RGB) |

### BatchNorm2d — the stabilizer

During training, numbers flowing through the network can explode or vanish, stopping learning.
BatchNorm looks at the current batch, computes mean and standard deviation for each channel,
and rescales values to stay in a healthy range. It acts like a thermostat for activations.

For a batch of $m$ values $x_1 \dots x_m$ in one channel:

$$\mu = \frac{1}{m}\sum x_i \qquad \text{(mean)}$$
$$\sigma^2 = \frac{1}{m}\sum (x_i - \mu)^2 \qquad \text{(variance)}$$
$$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \varepsilon}} \qquad \text{(normalize)}$$
$$y_i = \gamma \cdot \hat{x}_i + \beta \qquad \text{(learned scale and shift)}$$

$\gamma$ and $\beta$ are learnable parameters — BatchNorm can undo the normalization if that
turns out to be harmful for a particular channel. $\varepsilon$ is a tiny constant (1e-5) to avoid
division by zero.

The `num_features` argument must match the `out_channels` of the layer right before it.

### Why `bias=False` on every ConvTranspose2d

BatchNorm has its own learned bias internally. If the convolution also has a bias,
they fight each other. Removing the conv's bias avoids the conflict and saves memory.

### ReLU and Tanh — the activation functions

- **ReLU**: clips negative values to 0. Keeps the signal clean. Used after every BatchNorm.
- **Tanh**: squashes outputs to [-1, 1]. Used only on the final layer, because images are
  normalized to [-1, 1] by the dataset transform — they must match.

### Why Tanh and not something else?

The dataset transform `Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))` maps raw [0, 1] pixels
to [-1, 1]. The generator must output values in that same range:

| Function | Output range | Problem |
|---|---|---|
| **Tanh** | [-1, 1] | Matches perfectly |
| Sigmoid | [0, 1] | Generator can never produce dark pixels (negative values). |
| ReLU | [0, ∞) | No upper bound. Output explodes. |
| No activation | (-∞, ∞) | Completely unbounded. |

Tanh is the only common activation that naturally lives in [-1, 1]. The generator and
the dataset must speak the same numeric language.

### Why this three-part pattern?

Every generator layer repeats the same sandwich:

```
ConvTranspose2d  →  BatchNorm2d  →  ReLU
    (transform)       (stabilize)     (nonlinearity)
```

| Step | What it does | What happens without it |
|---|---|---|
| ConvTranspose2d | Learns spatial patterns | Nothing. It's the core engine. |
| BatchNorm2d | Rescales to mean=0, std=1 | Gradients vanish or explode. |
| ReLU | Clips negatives to 0. Adds nonlinearity | Stack collapses to one linear layer. |

### What does "nonlinearity" actually mean?

A **linear** operation is just scaling and adding: `y = 2·x + 1`. If you stack two linear
layers without any nonlinearity in between, the math collapses them into a single equivalent
layer. 1000 linear layers = 1 linear layer. No extra power.

A **nonlinear** operation like ReLU breaks the chain. Clipping negatives to zero means the
next Conv sees a genuinely different shape, so each layer can learn progressively more
complex patterns. Deep networks without activations are like a tall stack of paper —
it's just the same sheet with extra steps. Deep networks with ReLUs are like folding
the paper into a shape.

### State size comments

Comments like `# state size: (NGF*4) x 8 x 8` mean:
```
channels=256, height=8, width=8   (batch dimension omitted)
```
The full tensor shape is `(BATCH_SIZE, 256, 8, 8)`.

In [ ]:
# Generator Code
# Source: https://docs.pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html


class Generator(torch.nn.Module):
    def __init__(self, ngpu: int) -> None:
        super().__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d(
                in_channels=NUMBER_Z_DIMENSIONS,
                out_channels=NUMBER_GENERATOR_FEATURES * 8,
                kernel_size=4,
                stride=1,
                padding=0,
                bias=False,
            ),
            nn.BatchNorm2d(num_features=NUMBER_GENERATOR_FEATURES * 8),
            nn.ReLU(inplace=True),
            # state size: (NUMBER_GENERATOR_FEATURES*8) x 4 x 4
            nn.ConvTranspose2d(
                in_channels=NUMBER_GENERATOR_FEATURES * 8,
                out_channels=NUMBER_GENERATOR_FEATURES * 4,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(num_features=NUMBER_GENERATOR_FEATURES * 4),
            nn.ReLU(inplace=True),
            # state size: (NUMBER_GENERATOR_FEATURES*4) x 8 x 8
            nn.ConvTranspose2d(
                in_channels=NUMBER_GENERATOR_FEATURES * 4,
                out_channels=NUMBER_GENERATOR_FEATURES * 2,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(num_features=NUMBER_GENERATOR_FEATURES * 2),
            nn.ReLU(inplace=True),
            # state size: (NUMBER_GENERATOR_FEATURES*2) x 16 x 16
            nn.ConvTranspose2d(
                in_channels=NUMBER_GENERATOR_FEATURES * 2,
                out_channels=NUMBER_GENERATOR_FEATURES,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(num_features=NUMBER_GENERATOR_FEATURES),
            nn.ReLU(inplace=True),
            # state size: (NUMBER_GENERATOR_FEATURES) x 32 x 32
            nn.ConvTranspose2d(
                in_channels=NUMBER_GENERATOR_FEATURES,
                out_channels=NUMBER_CHANNELS,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.Tanh(),
            # state size: (NUMBER_CHANNELS) x 64 x 64
        )

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        return self.main(input)

## Conv2d vs ConvTranspose2d — shrink vs grow

Both take the same arguments (kernel_size, stride, padding), but do opposite things:

| | Conv2d (discriminator) | ConvTranspose2d (generator) |
|---|---|---|
| Direction | Shrinks: 64×64 → 1×1 | Grows: 1×1 → 64×64 |
| Stride 2 | Output is half the input size | Output is double the input size |
| Mental model | Zooming out, summarizing | Painting details between pixels |
| Can it grow? | No — max preserves size with stride=1 | Yes — its purpose |

Conv2d with stride=1 and proper padding can preserve size (8×8 → 8×8), but it can never increase it.
Growing requires either ConvTranspose2d or Upsample + Conv2d.

### LeakyReLU vs ReLU — the discriminator's choice

| | ReLU | LeakyReLU |
|---|---|---|
| Negative inputs | 0 (dead neuron) | 0.2 × input (tiny negative signal) |
| Used in | Generator | Discriminator |
| Why | Simple, clean signal | Prevents dead gradients in adversarial training |

A regular ReLU clips negatives to zero — if a neuron goes negative, it stops learning forever.
LeakyReLU lets 20% of the negative signal through, so neurons can recover later.

## Discriminator — the image shrinker

The discriminator is a binary classifier. It takes a 64×64 RGB image and compresses it
through five Conv2d layers into a single number between 0 and 1:

| Layer | Shape out | Channels |
|---|---|---|
| Input image | 64×64 | 3 (RGB) |
| Conv2d (stride=2) | 32×32 | 64 |
| Conv2d (stride=2) | 16×16 | 128 |
| Conv2d (stride=2) | 8×8 | 256 |
| Conv2d (stride=2) | 4×4 | 512 |
| Conv2d (stride=1) + Sigmoid | 1×1 | 1 (real/fake score) |

### Generator vs Discriminator — mirror images

| | Generator | Discriminator |
|---|---|---|
| Core layer | ConvTranspose2d (grows) | Conv2d (shrinks) |
| Channels trend | 512 → 256 → 128 → 64 → 3 | 3 → 64 → 128 → 256 → 512 → 1 |
| Spatial trend | 1×1 → 64×64 | 64×64 → 1×1 |
| Activation (hidden) | ReLU | LeakyReLU (prevents dead neurons) |
| Activation (final) | Tanh (output [-1, 1]) | Sigmoid (output [0, 1]) |
| First layer BatchNorm? | Yes | No (paper recommendation) |

The first discriminator layer skips BatchNorm. The DCGAN paper found that normalizing the
raw input image hurts the discriminator's ability to detect real vs fake — it needs the raw
pixel statistics to make that judgment. Later layers add BatchNorm for stability.

### Sigmoid — the final gate

The last layer uses Sigmoid to squash the output to [0, 1]:
- Output ≈ 1 → "looks real"
- Output ≈ 0 → "definitely fake"
- Output ≈ 0.5 → "can't tell" (the equilibrium goal)

### Why Conv2d → Sigmoid instead of Flatten → Linear → Sigmoid?

Both approaches turn the final (512, 4, 4) feature map into a single probability.
The difference is style, not function:

| | Conv2d → Sigmoid (DCGAN) | Flatten → Linear → Sigmoid |
|---|---|---|
| Final step | 4×4 kernel slides over 4×4 input → 1 | Flatten 8192 numbers, multiply by weights |
| Parameters | 512 × 4 × 4 = ~8,192 (no bias) | 8192 × 1 + 1 bias = 8,193 |
| Result | Nearly identical math, one extra parameter | Nearly identical math, one extra parameter |
| Why DCGAN chose it | Keeps the discriminator fully convolutional end-to-end | Traditional classifier pattern |

With `kernel_size=4, padding=0` on a 4×4 input, Conv2d(512, 1) is mathematically identical
to Flatten + Linear(8192, 1). The DCGAN paper's philosophy was "keep everything convolutional"
— no Linear layers, no pooling. The final Conv2d is a stylistic choice that flows from
that principle, not a functional requirement.

In [ ]:
# Discriminator Code
# Source: https://docs.pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html


class Discriminator(torch.nn.Module):
    def __init__(self, ngpu: int) -> None:
        super().__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            # input is (NUMBER_CHANNELS) x 64 x 64
            nn.Conv2d(
                in_channels=NUMBER_CHANNELS,
                out_channels=NUMBER_DISCRIMINATOR_FEATURES,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
            # state size: (NUMBER_DISCRIMINATOR_FEATURES) x 32 x 32
            nn.Conv2d(
                in_channels=NUMBER_DISCRIMINATOR_FEATURES,
                out_channels=NUMBER_DISCRIMINATOR_FEATURES * 2,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(num_features=NUMBER_DISCRIMINATOR_FEATURES * 2),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
            # state size: (NUMBER_DISCRIMINATOR_FEATURES*2) x 16 x 16
            nn.Conv2d(
                in_channels=NUMBER_DISCRIMINATOR_FEATURES * 2,
                out_channels=NUMBER_DISCRIMINATOR_FEATURES * 4,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(num_features=NUMBER_DISCRIMINATOR_FEATURES * 4),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
            # state size: (NUMBER_DISCRIMINATOR_FEATURES*4) x 8 x 8
            nn.Conv2d(
                in_channels=NUMBER_DISCRIMINATOR_FEATURES * 4,
                out_channels=NUMBER_DISCRIMINATOR_FEATURES * 8,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(num_features=NUMBER_DISCRIMINATOR_FEATURES * 8),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
            # state size: (NUMBER_DISCRIMINATOR_FEATURES*8) x 4 x 4
            nn.Conv2d(
                in_channels=NUMBER_DISCRIMINATOR_FEATURES * 8,
                out_channels=1,
                kernel_size=4,
                stride=1,
                padding=0,
                bias=False,
            ),
            nn.Sigmoid(),
            # output: scalar probability (1 x 1 x 1)
        )

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        return self.main(input)

## Creating the models

### `weights_init` — DCGAN paper recipe

The DCGAN paper specifies all weights must start from `Normal(mean=0, std=0.02)`.
PyTorch's default init is different (Kaiming uniform), so we override it with `.apply()`.

### `DataParallel` — multi-GPU wrapper (optional)

With `NGPU=1` this is a no-op — the `if` condition is false. If you have multiple GPUs,
`DataParallel` splits the batch across them and recombines results.

### `.to(device)` — move to GPU

Model parameters and input tensors must live on the same device.

### `print(netG)` — sanity check

Printing the model confirms the layer structure matches the shape progression table.

In [ ]:
# Create the Generator

netG = Generator(ngpu=NGPU).to(device)
if (device.type == "cuda") and (NGPU > 1):
    netG = torch.nn.DataParallel(netG, list(range(NGPU)))
netG.apply(weights_init)
print(netG)

In [ ]:
# Create the Discriminator

netD = Discriminator(ngpu=NGPU).to(device)
if (device.type == "cuda") and (NGPU > 1):
    netD = torch.nn.DataParallel(netD, list(range(NGPU)))
netD.apply(weights_init)
print(netD)

## Loss function and optimizers

### `BCELoss` — Binary Cross Entropy

Measures how far a predicted probability (0–1) is from the true label (0 or 1).

### `FIXED_NOISE` — the progress camera

A batch of 64 latent vectors saved once and reused every epoch. By feeding the same
noise each time, you see how the *same recipes* improve as training progresses.

### `REAL_LABEL` / `FAKE_LABEL` — ground truth

Real images = 1.0, fake images = 0.0. Makes the training loop readable.

### Adam optimizer — DCGAN paper settings

| Parameter | Value | Why |
|---|---|---|
| lr | 0.0002 | Smaller than typical (0.001) — GANs are sensitive |
| beta1 | 0.5 | Lower than default (0.9) — reduces momentum for stability |
| beta2 | 0.999 | Default value |

Each network gets its own optimizer because they learn different things at
different rates. Sharing would couple their updates and destabilize training.

In [ ]:
# Loss function, fixed noise, labels, and optimizers
# Source: https://docs.pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html

criterion = nn.BCELoss()

FIXED_NOISE = torch.randn(64, NUMBER_Z_DIMENSIONS, 1, 1, device=device)

REAL_LABEL = 1.0
FAKE_LABEL = 0.0

optimizerD = optim.Adam(netD.parameters(), lr=LR, betas=(BETA_1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=LR, betas=(BETA_1, 0.999))

## Training loop — how the code maps to the GAN formula

The Goodfellow (2014) minimax value function:

$$\min_G \max_D \; \underbrace{\mathbb{E}_{x \sim p_{\text{data}}}[\log D(x)]}_{\text{D gets real right}} + \underbrace{\mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]}_{\text{D spots fakes}}$$

### Term 1 — `log D(x)` (D correctly calls real images "real")

```python
real = data[0].to(device)               # x ~ p_data
label = ... REAL_LABEL ...              # 1.0
output = netD(real).view(-1)            # D(x)
errD_real = criterion(output, label)     # BCELoss(label=1) = -log(D(x))
```
Minimizing `-log(D(x))` = maximizing `log(D(x))`. D's first half of `max_D`.

### Term 2 — `log(1 - D(G(z)))` (D correctly calls fakes "fake")

```python
noise = torch.randn(...)                # z ~ p_z
fake = netG(noise)                      # G(z)
label.fill_(FAKE_LABEL)                 # 0.0
output = netD(fake.detach()).view(-1)   # D(G(z)), .detach() freezes G
errD_fake = criterion(output, label)     # BCELoss(label=0) = -log(1-D(G(z)))
```
Minimizing `-log(1-D(G(z)))` = maximizing `log(1-D(G(z)))`. D's second half of `max_D`.

### G's update — the label-flip trick (non-saturating loss)

The formula says G wants to **minimize** `log(1 - D(G(z)))`. But when D is winning,
`D(G(z)) ≈ 0`, so `log(1-0) = 0` — flat gradient, G can't learn.
The fix: flip the label and minimize `-log(D(G(z)))` instead:

```python
netG.zero_grad()
label.fill_(REAL_LABEL)                 # 1.0 — the label flip
output = netD(fake).view(-1)            # D(G(z)) — NO .detach(), gradients flow to G
errG = criterion(output, label)          # BCELoss(label=1) = -log(D(G(z)))
errG.backward()
optimizerG.step()
```
G pushes `D(G(z))` toward 1 — D is fooled. This has better gradients than the
original `log(1-D(G(z)))` when G is weak.

### Summary

| Math | Code | Who |
|---|---|---|
| `max log D(x)` | `criterion(D(real), 1)` → min `-log D(x)` | D |
| `max log(1 - D(G(z)))` | `criterion(D(fake.detach()), 0)` → min `-log(1-D(G(z)))` | D |
| `min -log(D(G(z)))` | `criterion(D(fake), 1)` → min `-log D(G(z))` | G |

In [ ]:
# Training Loop
# Source: https://docs.pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html

# Lists to track progress
img_list: list = []
g_losses: list[float] = []
d_losses: list[float] = []
iters = 0

print("Starting Training Loop...")
for epoch in range(NUM_EPOCHS):
    for i, data in enumerate(dataloader, 0):
        # ============================================
        # (1) Update D: maximize log(D(x)) + log(1 - D(G(z)))
        # ============================================
        netD.zero_grad()

        # Train with real images
        real = data[0].to(device)
        b_size = real.size(0)
        label = torch.full((b_size,), REAL_LABEL, dtype=torch.float, device=device)
        output = netD(real).view(-1)
        errD_real = criterion(output, label)
        errD_real.backward()

        # Train with fake images
        noise = torch.randn(b_size, NUMBER_Z_DIMENSIONS, 1, 1, device=device)
        fake = netG(noise)
        label.fill_(FAKE_LABEL)
        output = netD(fake.detach()).view(-1)
        errD_fake = criterion(output, label)
        errD_fake.backward()
        errD = errD_real + errD_fake
        optimizerD.step()

        # ============================================
        # (2) Update G: maximize log(D(G(z)))
        # ============================================
        netG.zero_grad()
        label.fill_(REAL_LABEL)  # fake labels are real for generator cost
        output = netD(fake).view(-1)
        errG = criterion(output, label)
        errG.backward()
        optimizerG.step()

        # Save losses for plotting
        g_losses.append(errG.item())
        d_losses.append(errD.item())

        # Check progress and save generated images
        if (iters % 500 == 0) or ((epoch == NUM_EPOCHS - 1) and (i == len(dataloader) - 1)):
            with torch.inference_mode():
                fake = netG(FIXED_NOISE).detach().cpu()
            img_list.append(vutils.make_grid(fake, padding=2, normalize=True))

        iters += 1

    print(f"[{epoch + 1}/{NUM_EPOCHS}] complete")